# EXP-2026-007 / Q5-D — PREP_DATA-A (quest47)

## `EXP-2026-007 / PREP_DATA-A ACQUIRE ONLY`
## `NO TRAINING / NO SCIENTIFIC ANALYSIS`
## `PREP_DATA RESULT NOT RUN`

**아래 셀을 실행하기 전까지 이 문서의 어떤 숫자도 결과가 아니다.**

- 이번에 하는 일: PhysioNet **버전 고정** 자산(`pwave 1.0.0` · `mitdb 1.0.0`)을 Google Drive에
  **불변 자산**으로 확보하고, publisher hash와 WFDB open으로 검증한 뒤 **중단**한다.
- 현재 mode: 아래 셀 1의 `MODE` (기본값 `DESIGN`).
- 대상 Drive 경로: `MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/`
  (`source/pwave-1.0.0/` · `source/mitdb-1.0.0/` · `audit/`)
- 예상 데이터 용량: **mitdb ~104.3 MB · pwave ~22.4 MB**. 이 수치는 hard gate가 아니라
  누락·폭증을 발견하기 위한 감사 정보다 — 최종 기준은 publisher manifest와 **파일별 hash**다.
- **기존 파일·폴더를 삭제하거나 덮어쓰지 않는다.** 최종 경로가 이미 있으면 먼저 모든 hash를
  검증하고, 같은 경로에 내용이 다르면 `EXISTING_ASSET_CONFLICT`로 중단한다. 누락 파일은
  조용히 채우지 않고 **timestamped staging 경로**에 받은 뒤 partial 상태로 보고한다.
- 받는 것: mitdb 공식 `RECORDS` **48개 전체** × `.dat/.hea/.atr`,
  pwave **12개**(`100 101 103 106 117 119 122 207 214 222 223 231`) × `.dat/.hea/.pwave`,
  그리고 두 DB의 `RECORDS` · `ANNOTATORS` · `SHA256SUMS.txt`.

### 이 단계에서 하지 않는 것 (전부 금지)

P-wave delineation · 측정 자격검증 · P-to-R/RR/coupling 계산 · DS2 label과 V10 확률 열람 ·
S PR-AUC 등 과학 metric · SHAM permutation · 모델 학습/미세조정/체크포인트 ·
기존 Drive 자산과 run bundle 덮어쓰기.

### 판정 (셋 중 정확히 하나)

| 코드 | 뜻 |
|---|---|
| `PREP_DATA_ACQUIRED_VERIFIED` | 48/48 + 12/12 전부 hash·WFDB 통과, 기존 자산 무변경 |
| `INPUT_ABSENT_OR_MISMATCH` | 누락·hash 불일치·record 목록 불일치·WFDB 실패·partial conflict 중 하나라도 |
| `PREP_DATA_RESULT_NOT_RUN` | 코드만 있고 아직 Colab에서 실행하지 않음 |

**이번 판정은 데이터 준비 결과이지 EXP-2026-007의 과학적 `MEASURED` 판정이 아니다.
다음 단계는 자동 실행되지 않는다.**

### Colab 실행 순서

셀 1은 mode를 정하는 설정 셀이다. **건너뛰고 셀 2부터 실행해도 된다** — 그때는 셀 2가
`MODE="DESIGN"` · `BRANCH`(이 브랜치)를 기본값으로 채운다. mode를 바꿀 때만 셀 1을 고쳐서
실행한 뒤 해당 mode 셀로 간다.

1. 셀 2 — repo 준비와 commit SHA 확인 (회귀 테스트 포함)
2. 셀 3 — Google Drive mount
3. 셀 1의 `MODE = "DESIGN"` 그대로 셀 4 실행 → source·예상 용량·Drive 경로·금지사항 확인
4. 예상 경로와 용량 확인
5. 셀 1에서 `MODE = "PREP_DATA_ACQUIRE"` 로 바꾸고 셀 1→5 실행
6. 판정과 checksum gate 확인
7. 셀 1에서 `MODE = "PREP_DATA_REPORT"` 로 바꾸고 셀 1→3→6 실행 (저장 bundle 재표시)
8. 출력을 포함한 채로 notebook 저장
9. **여기서 중단하고 결과를 보고한다.**


In [23]:
# ── 셀 1: 실행 설정 (정확히 하나의 mode) ─────────────────────────────────────
VALID_MODES = ("DESIGN", "PREP_DATA_ACQUIRE", "PREP_DATA_REPORT")
MODE = "DESIGN"                    # DESIGN -> PREP_DATA_ACQUIRE -> PREP_DATA_REPORT
assert MODE in VALID_MODES, f"MODE must be one of {VALID_MODES}"

BRANCH = "claude/exp-2026-007-prep-data-a-jruqd5"   # 이 notebook이 쓸 repo 브랜치
NEED_Q5D = 2                                        # 요구하는 최소 모듈 버전

print("EXP-2026-007 / PREP_DATA-A ACQUIRE ONLY")
print("NO TRAINING / NO SCIENTIFIC ANALYSIS")
print("mode:", MODE)


EXP-2026-007 / PREP_DATA-A ACQUIRE ONLY
NO TRAINING / NO SCIENTIFIC ANALYSIS
mode: DESIGN


In [16]:
# ── 셀 2: repo 준비 + commit SHA + 회귀 테스트 ───────────────────────────────
import os, subprocess, sys

# 셀 1을 건너뛰고 여기서부터 실행해도 돌아가도록 기본값을 쓴다.
# 셀 1을 실행했다면 거기서 정한 값이 그대로 유지된다.
BRANCH = globals().get("BRANCH", "claude/exp-2026-007-prep-data-a-jruqd5")
NEED_Q5D = int(globals().get("NEED_Q5D", 2))
MODE = globals().get("MODE", "DESIGN")
print("branch:", BRANCH, "| mode:", MODE)

REPO = "/content/my-github-test"
os.chdir("/content")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone",
                    "https://github.com/ehdbddl06001-ui/my-github-test.git"],
                   check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"],
               check=True)

for _m in [m for m in list(sys.modules) if m.startswith("q5d_")]:
    del sys.modules[_m]
sys.path.insert(0, os.path.join(REPO, "mit-bih"))
import q5d_expert_validated_pwave_timing as Q5D
assert Q5D.MODULE_VERSION >= NEED_Q5D, "stale module — 이 셀을 다시 실행"
print("module:", Q5D.MODULE_VERSION, Q5D.MODULE_BUILD, "|", Q5D.STATUS)

_rc = subprocess.run(
    [sys.executable, "mit-bih/test_q5d_expert_validated_pwave_timing.py"],
    cwd=REPO).returncode
assert _rc == 0, "테스트 실패 — 다운로드를 시작하지 않는다"


branch: claude/exp-2026-007-prep-data-a-jruqd5 | mode: DESIGN
commit: 02a82bb564170558afa74a0b19715cdbc54315e1
module: 3 2026-08-09 | PREP_DATA DESIGN / RESULT NOT RUN


In [17]:
# ── 셀 3: Google Drive mount + 대상 경로 ─────────────────────────────────────
import os
assert "Q5D" in globals(), "먼저 셀 2(repo 준비)를 실행한다"
MODE = globals().get("MODE", "DESIGN")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive"
ASSET_ROOT = os.path.join(DRIVE_ROOT, Q5D.DRIVE_ASSET_REL)
print("asset root :", ASSET_ROOT)
print("존재 여부   :", os.path.isdir(ASSET_ROOT))

_state = Q5D.report_bundle(ASSET_ROOT)
print("현재 상태   :", _state["decision"])
if _state["decision"] == Q5D.DECISION_NOT_RUN:
    print("→ PREP_DATA RESULT NOT RUN (아직 아무것도 내려받지 않았다)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
asset root : /content/drive/MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data
존재 여부   : True
현재 상태   : PREP_DATA_ACQUIRED_VERIFIED


In [18]:
# ── 셀 4: DESIGN — source·용량·경로·금지사항만 표시 (다운로드 없음) ──────────
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "DESIGN", f"이 셀은 DESIGN 전용 (지금 {MODE} — 셀 1에서 바꾼다)"

print(Q5D.design_card(ASSET_ROOT, MODE))
for _s in Q5D.SOURCES:
    print(f"{_s.db} {_s.version}: {len(_s.records_expected)} records x "
          f"{len(_s.extensions)} ext = "
          f"{len(_s.records_expected) * len(_s.extensions)} files + "
          f"{len(Q5D.REQUIRED_METADATA)} metadata | DOI {_s.doi} | "
          f"license {_s.license_name}")
print("허용 mode :", Q5D.MODES)
print("차단 mode :", Q5D.FORBIDDEN_MODES, "→ 호출하면 중단된다")
print("하지 않는 것:", Q5D.build_config()["not_performed"])


  EXP-2026-007 / PREP_DATA-A ACQUIRE_ONLY
  NO TRAINING / NO SCIENTIFIC ANALYSIS
  mode           : DESIGN
  status         : PREP_DATA DESIGN / RESULT NOT RUN
  drive target   : /content/drive/MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data
  source         : mitdb 1.0.0 · DOI 10.13026/C2F305 · 48 records · ~104.3 MB · https://physionet.org/files/mitdb/1.0.0/
  source         : pwave 1.0.0 · DOI 10.13026/C2108F · 12 records · ~22.4 MB · https://physionet.org/files/pwave/1.0.0/
  extensions     : mitdb .dat/.hea/.atr · pwave .dat/.hea/.pwave
  immutability   : 기존 파일·폴더를 덮어쓰거나 삭제하지 않는다. 같은 경로에 내용이 다르면 EXISTING_ASSET_CONFLICT로 중단한다.
  NOT performed  : delineation · P-to-R · RR/coupling · DS2 outcome · S PR-AUC · SHAM permutation · training
  decision codes : PREP_DATA_ACQUIRED_VERIFIED | INPUT_ABSENT_OR_MISMATCH | PREP_DATA_RESULT_NOT_RUN
------------------------------------------------------------------------
  Colab 실행 순서
   1) repo 준비 + commit SHA 확인
   2) Google Drive mount
  

In [20]:
# ── 셀 5: PREP_DATA_ACQUIRE — 다운로드 + checksum + WFDB 검증 ───────────────
import time
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "PREP_DATA_ACQUIRE", \
    f"이 셀은 PREP_DATA_ACQUIRE 전용 (지금 {MODE} — 셀 1에서 바꾼 뒤 셀 1을 실행)"

TS = time.strftime("%Y%m%dT%H%M%S")
_log = Q5D.RunLog()
OUT = Q5D.run_prep_data_acquire(ASSET_ROOT, TS, log=_log,
                                progress=Q5D.progress_printer())

print()
print(Q5D.render_gate_card(OUT["decision"], OUT["inventories"], OUT["wfdb"]))
print()
print("audit bundle :", OUT["audit_dir"])
print("이번 실행 사본:", OUT["run_dir"])
print("staging      :", OUT["staging_root"])
print("observed     : %.1f MB" % (OUT["result"]["observed_total_bytes"] / 1e6))

_recs = Q5D.record_inventory(OUT["inventories"], OUT["wfdb"], OUT["duplicates"])
_bad = [r for r in _recs if r["status"] != Q5D.FILE_VERIFIED]
print(f"record 표: {len(_recs)}행 · 실패 {len(_bad)}행")
for _r in _bad[:40]:
    print("  FAIL", _r["database"], _r["record"], _r["status"], _r["detail"])
print("다음 단계(자격검증·DS2 분석·학습)는 실행되지 않았다 — 사용자 승인 대상.")


[    0.0s] EXP-2026-007 / PREP_DATA-A ACQUIRE_ONLY — NO TRAINING / NO SCIENTIFIC ANALYSIS
[    0.0s] mitdb 1.0.0 — read /content/drive/MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/source/mitdb-1.0.0 (pre-existing, immutable)
[    0.0s]   new files (if any) go to staging: /content/drive/MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/staging_20260809T153151/source/mitdb-1.0.0
[    0.0s]   SHA256SUMS.txt: 704 publisher entries (sha256 b61158a96d5f2ca8…)
[    0.1s]   RECORDS declares 48 record(s); expected 48
[    0.1s]   144 required record file(s) (48 records x 3 ext)
  [##......................] file mitdb 12/144 103.atr VERIFIED
  [####....................] file mitdb 24/144 107.atr VERIFIED
  [######..................] file mitdb 36/144 112.atr VERIFIED
  [########................] file mitdb 48/144 116.atr VERIFIED
  [##########..............] file mitdb 60/144 121.atr VERIFIED
  [############............] file mitdb 72/144 200.atr VERIFIED
  [##############........

In [22]:
# ── 셀 6: PREP_DATA_REPORT — 저장된 bundle만 다시 표시 (재계산 없음) ────────
assert "ASSET_ROOT" in globals(), "먼저 셀 2·3을 실행한다"
MODE = globals().get("MODE", "DESIGN")
assert MODE == "PREP_DATA_REPORT", \
    f"이 셀은 PREP_DATA_REPORT 전용 (지금 {MODE} — 셀 1에서 바꾼 뒤 셀 1을 실행)"

REP = Q5D.report_bundle(ASSET_ROOT)
print("decision   :", REP["decision"])
print("recomputed :", REP["recomputed"])
print("누락 audit :", REP.get("missing_audit_files"))
print("보관된 실행:", REP.get("archived_runs"))
print()
print(REP.get("summary", "(no summary — 아직 실행되지 않았다)"))
if REP.get("decision_detail"):
    print(Q5D.render_gate_card(REP["decision_detail"]))


decision   : PREP_DATA_ACQUIRED_VERIFIED
recomputed : False
누락 audit : []
보관된 실행: ['20260809T151626', '20260809T153151']

# EXP-2026-007 / PREP_DATA-A ACQUIRE_ONLY

- **NO TRAINING / NO SCIENTIFIC ANALYSIS**
- 판정: **PREP_DATA_ACQUIRED_VERIFIED**  (12/12 gate pass)
- 이 판정은 **데이터 준비 결과**이고 EXP-2026-007의 과학적 판정이 아니다.

## 소스

- `mitdb 1.0.0` DOI `10.13026/C2F305` · https://physionet.org/files/mitdb/1.0.0/ · license Open Data Commons Attribution License v1.0
  - RECORDS 48개 (기대 48개, 일치 True) · checksum pass 1.000 (147/147) · 관측 93.9 MB (공개 트리 참고치 ~104.3 MB)
  - SHA256SUMS.txt sha256 `b61158a96d5f2ca80edfb354a9a66a6324836c390a84e1966dcee2b907d6be43`
- `pwave 1.0.0` DOI `10.13026/C2108F` · https://physionet.org/files/pwave/1.0.0/ · license Open Data Commons Attribution License v1.0
  - RECORDS 12개 (기대 12개, 일치 True) · checksum pass 1.000 (39/39) · 관측 23.5 MB (공개 트리 참고치 ~22.4 MB)
  - SHA256SUMS.txt sha256 `f4c7f0905f07fa63e1fe1561292e2f8f9cf00e869fa69ddf99c2d1ad151c94e1`

## WFDB open

- 60/60 

## 마지막 화면 — 지금 상태와 다음에 할 일

- `EXP-2026-007 / PREP_DATA-A ACQUIRE ONLY` · `NO TRAINING / NO SCIENTIFIC ANALYSIS`
- 셀 5를 실행하지 않았다면 상태는 **`PREP_DATA RESULT NOT RUN`** 이다. 실행하지 않은
  notebook을 `ACQUIRED_VERIFIED` 나 `MEASURED` 로 표시하지 않는다.
- 저장 위치: `MyDrive/MedKOS/ecg-model/assets/EXP-2026-007_prep_data/`
  · 자산 `source/mitdb-1.0.0/` · `source/pwave-1.0.0/`
  · 감사 `audit/config.json · asset_manifest.json · source_inventory.csv ·
  checksum_report.csv · record_inventory.csv · wfdb_open_report.csv ·
  decision.json · log.txt · summary.md`
- 실패했다면 `decision.json` 의 `first_stopping_reason` 과 `checksum_report.csv` 의
  해당 행(예상 hash · 관측 hash · 파일명)을 그대로 보고한다. 더 강한 명령으로 지우거나
  무조건 다시 받지 않는다.

### 실행 순서 (다시)

1. repo 준비와 commit SHA 확인 → 2. Drive mount → 3. mode `DESIGN` 실행 →
4. 예상 경로·용량 확인 → 5. mode `PREP_DATA_ACQUIRE` 로 전체 실행 →
6. 판정과 checksum gate 확인 → 7. mode `PREP_DATA_REPORT` 로 저장 bundle 재표시 →
8. 출력 포함 notebook 저장 → 9. **중단하고 사용자에게 보고**

### 여기서 멈춘다

P-wave delineation, 측정 자격검증, beat join, DS2 outcome 분석, S PR-AUC, SHAM
permutation, 모델 학습은 이 notebook에 **경로 자체가 없다**. 다음 substage는 이
acquisition bundle을 사람이 검토하고 별도로 승인한 뒤에만 시작한다.
